# Download US County Demographics & Geography

Downloads racial demographic data for ALL US counties from the Census API (ACS 5-year 2023)
and county boundary geometry, saving as JSON/TopoJSON files.

**Requirements:**
```bash
pip install requests python-dotenv geopandas topojson
```

**Setup:**
Create a `.env` file in the project root with:
```
census_api_key=YOUR_API_KEY_HERE
```
Get a free key at: https://api.census.gov/data/key_signup.htm

In [ ]:
import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv
import geopandas as gpd
import topojson as tp

# Load API key from .env file
load_dotenv()
census_api_key = os.getenv("census_api_key")

if not census_api_key:
    raise ValueError("Missing census_api_key in .env file!")

# Output directory
PUBLIC_DIR = (Path.cwd().parent / "public").resolve()
print(f"Output directory: {PUBLIC_DIR}")

# Census API base URL
API_URL = "https://api.census.gov/data/2023/acs/acs5"

# Variables to fetch (ACS table B03002 = Hispanic/Latino Origin by Race)
VARIABLES = {
    "B03002_001E": "total_pop",
    "B03002_003E": "white_non_hisp",
    "B03002_004E": "black_non_hisp",
    "B03002_005E": "native_am_non_hisp",
    "B03002_006E": "asian_non_hisp",
    "B03002_012E": "hisp_any_race",
    "B03002_013E": "white_hisp",
}


In [ ]:
# ============================================================
# US COUNTY DEMOGRAPHICS (ALL STATES)
# ============================================================
print("Downloading US county demographics (all states)...")

response = requests.get(API_URL, params={
    "get": "NAME," + ",".join(VARIABLES.keys()),
    "for": "county:*",
    "key": census_api_key,
})
response.raise_for_status()
data = response.json()

# Convert to list of dicts
county_data = []
headers = data[0]
for row in data[1:]:
    record = {"GEOID": row[headers.index("state")] + row[headers.index("county")]}
    record["name"] = row[headers.index("NAME")]
    for var_code, var_name in VARIABLES.items():
        val = row[headers.index(var_code)]
        record[var_name] = int(val) if val and val != "-666666666" else 0
    county_data.append(record)

# Save to JSON
with open(PUBLIC_DIR / "US_county_demographics.json", "w") as f:
    json.dump(county_data, f)

print(f"  ✅ Saved {len(county_data)} counties to US_county_demographics.json")


In [ ]:
# ============================================================
# US COUNTY GEOGRAPHY
# ============================================================
print("Downloading US county geography...")

# Census Bureau Cartographic Boundary URL (2023, 500k resolution)
COUNTY_URL = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_500k.zip"

gdf_county = gpd.read_file(COUNTY_URL)
print(f"  Downloaded {len(gdf_county)} counties")
print(f"  Columns: {list(gdf_county.columns)}")

# Convert to TopoJSON (no filtering - keep all US counties)
topo_county = tp.Topology(gdf_county, prequantize=True, object_name="county")
output_path = PUBLIC_DIR / "US_county_geo.topojson"
with open(output_path, "w") as f:
    f.write(topo_county.to_json())
print(f"  Saved to {output_path}")
print(f"  Size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")


In [ ]:
# ============================================================
# DONE - Show sample output
# ============================================================
print("\n✅ All done! Created:")
print("  - US_county_demographics.json")
print("  - US_county_geo.topojson")

print("\nSample county record:")
print(json.dumps(county_data[0], indent=2))

print(f"\nTotal US counties: {len(county_data)}")
